# 排查顺序的核心逻辑
- 是“先排除确定性故障，再处理概率性难题”：
  - 工程层问题具有二元可判定性，可通过检查直接确认或排除；
  - <font color='red'>策略坍塌属于灰度诊断，需排除工程干扰后才能归因</font>
  
具体解释如下：

## 🔧 工程层问题：二元可判定，排查成本低

<font color='red'>工程层故障具有“要么有、要么没有”的二元特征</font>，可通过明确检查项直接确认或排除：
  - 参数冻结：
    - LoRA参数被静默注册为不可训练，梯度为零。系统仍在计算损失，日志看似正常，但策略分布从未改变。只需记录可训练参数数量、抽查关键层梯度范数即可确认。
  - 数值溢出：
    - 低精度（如bfloat16）下，PPO重要性比率在早期就超过百万甚至产生NaN/Inf。监控比率最大值、分位数和NaN计数即可定位。
  - 显存溢出（OOM）与梯度爆炸：
    - 属于基础设施层面的硬故障，报错信息明确，修复路径清晰（梯度累积、ZeRO分布式优化、梯度裁剪等）。
  - 数据与代码Bug：
    - 样本比例错误、特征缺失、数据路径配错等，几分钟即可验证。
  
## 🧠 策略坍塌类问题：灰度诊断，排查成本高
<font color='red'>策略坍塌本质是灰度问题</font>，表现为模型输出退化、分布失控、熵值骤降等，但往往没有明确报错：
  - 熵坍塌：
    - 策略概率分布从高熵退化为尖锐低熵，<font color='red'>模型失去探索性</font>
  - 奖励驱动坍塌：
    - 长尾奖励叠加KL项时，<font color='red'>少量异常样本制造过大优势估计</font>，策略一步跳到参考模型几乎不给概率的区域。
  - 梯度消失：
    - 同一Prompt下所有响应奖励一致，优势值归零，模型无法更新。
  - 价值震荡与Q值发散：
    - Critic网络估计不稳定，反向放大至Actor，导致训练瓦解。
  
## 📌 为何必须先排除工程问题
  1. 策略坍塌的表象极易被工程Bug伪装：
    - 参数冻结时loss曲线看似正常但策略未变；
    - 数值溢出早期loss平滑但比率已爆；
    - 梯度消失时reward平台期看似健康但模型已停止学习。
    <font color='red'>若跳过工程排查直接调策略，会将工程Bug误判为算法缺陷，浪费大量时间调参。</font>
    
  2. 工程问题修复成本低、确定性高：
    - 工程Bug修复路径明确（换精度、修代码、加监控），
    - 而策略坍塌涉及超参、奖励设计、KL系数等复杂权衡，且无唯一解。
    <font color='red'>先排除工程问题可避免在错误基础上做无效优化</font>
    
  3. 策略坍塌的诊断依赖干净的观测信号：
    - 策略坍塌诊断依赖梯度范数、熵值、优势分布等信号。若工程层存在溢出或冻结，这些信号本身就是被污染的，无法作为可靠诊断依据。

## 📋 实践中的标准排查清单
### 第一层：<font color='red'>工程层硬检查</font>
  - 可训练参数数量是否正确？关键层梯度范数是否非零？
  - 重要性比率最大值、分位数是否异常？有无NaN/Inf？
  - 显存是否溢出？梯度范数是否在合理范围？
    - <font color='red'>正常训练 0.1~1.0，警告区间 1.0~10.0，梯度爆炸 >10.0</font>
  - 数据样本比例、特征分布是否正确？
  
### 第二层：策略坍塌诊断（工程层排除干净后）
  - <font color='blue'>策略熵是否在持续下降？输出分布是否过度集中？</font>
  - 优势值分布是否合理？是否存在零方差组？
    - 零方差组是指同一 Prompt 下采样的所有候选回复奖励完全相同，导致组内奖励方差为零、优势值全部归零、无法产生有效梯度更新的样本组
    - <font color='red'>当组内所有回复的奖励完全相同时（比如全对或全错），标准差 σ = 0，优势值 A_i 全部为零，梯度消失，模型无法从这组样本中学到任何东西。</font>
  - 奖励信号是否稳定？是否存在**长尾异常值驱动训练**？
  - KL散度是否过大或过小？策略是否偏离参考模型太远？
  
### 核心原则：
  - “训练跑完”与“策略变好”是两道门
    - 先保证控制环在工程层面稳定运行，再判断任务收益是否真实存在。
    - <font color='red'>在工程层问题未排除前，任何策略层面的诊断结论都不可信</font>